# 도구 순서 설계 ② 하드 규칙 — Read→Edit 5겹 게이트 (GPT 버전)

**하드 규칙** = 순서를 어기면 되돌릴 수 없는 사고가 나는 지점에, 실행 직전 **코드가 사전조건을 검사**해 거부하는 장치. CC의 대표 사례인 Read→Edit은 단일 에러가 아니라 `readFileState` 상태머신 기반 **5겹 장치**다:

| # | 겹 | 내용 |
|---|---|---|
| 1 | **사전 경고** (description) | "편집 전에 read_file을 최소 한 번… **이 도구는 에러를 냅니다**" — 시도 전에 미리 알림 |
| 2 | **게이트1: 안 읽음** | 읽은 이력 없으면 거부. **부분읽기(offset/limit)는 읽은 걸로 안 쳐줌** |
| 3 | **게이트2: 낡은 읽기** | 읽은 뒤 파일 mtime이 더 최신이면 거부 → "다시 읽으세요" |
| 4 | **성공 후 자가갱신** | 편집 성공 시 timestamp 갱신 — 연속 edit이 게이트2에 안 걸리게 |
| 5 | **write에도 동일 게이트** | 기존 파일 덮어쓰기 전 같은 2중 검사 |

정확한 규칙 표현: "안 읽은 파일 금지"가 아니라 **"지금 디스크의 그 버전을 본 적 없으면 금지"** (신선도 추적).

**하드 규칙의 문법**: `설명문 사전경고("에러를 냅니다") + 실행 전 상태검사 + 에러를 tool_result로 반환(모델이 스스로 복구) + 성공 시 상태갱신`

GPT 이식 핵심: OpenAI 서버에는 "이 파일 읽었나" 같은 상태 개념이 없다 → **게이트는 반드시 클라이언트 실행기에** 심는다. 에러는 예외로 던지지 않고 `function_call_output`으로 돌려보낸다 — "에러 메시지는 로그가 아니라 프롬프트다."

> 소프트 규칙(입장권·탈출구·넛지)은 ① `cc_tool_sequence_soft_rules.ipynb` 참고.

In [1]:
import cc_tools as cts
from cc_tools import FS, reset_fs, external_modify, HardSession as Session, HARD_TOOLS as TOOLS, READ_FILE_STATE

# 소프트/하드 도구·세션은 한 파일 cc_tools.py에 함께 있다 (두 노트북이 같은 파일 사용).
# read/edit/write는 한 벌이고, 하드는 state=READ_FILE_STATE 게이트 스위치를 켠 것.
MODEL = cts.MODEL
print("기본 MODEL:", MODEL)

기본 MODEL: gpt-5-nano


## 1. 목 파일시스템 + `readFileState`

인메모리 목 파일시스템. mtime은 단조증가 카운터(CC도 정수 mtime 비교라 본질 동일). `external_modify()`가 "모델 모르게 파일이 바뀌는 상황"(사용자 편집·린터)을 시뮬레이션한다.

In [2]:
# 공통 목 코드베이스(orderhub, 40파일)로 초기화. mtime은 단조증가 카운터(CC도 정수 mtime 비교).
# external_modify()가 "모델 모르게 파일이 바뀌는 상황"(사용자 편집·린터)을 시뮬레이션한다.
# readFileState 상태머신(READ_FILE_STATE)은 세션 단위 — cc_tools.py 참고.
print(f"seeded {reset_fs()} files")

seeded 40 files


## 2. 게이트 구현

에러 문구는 CC 원문의 한국어 번역이다. 게이트 에러도 **정상 결과와 같은 채널**(`function_call_output`)로 반환된다 — 모델이 다음 라운드에 문구를 따라 스스로 복구한다.

In [3]:
# read_file·edit_file·write_file은 소프트와 '같은 한 벌'(cc_tools.py)을 쓰되,
# state=READ_FILE_STATE를 넘겨 readFileState 5겹 게이트를 켠다(소프트는 이 state를 안 넘겨
# 같은 함수가 게이트 없이 동작). 에러 문구는 CC 원문의 한국어 번역이며, 게이트 에러도 정상
# 결과와 같은 채널(function_call_output)로 반환된다 — 모델이 스스로 복구.
from cc_tools import (hard_read_file as read_file, hard_edit_file as edit_file,
                              hard_write_file as write_file, HARD_TOOL_IMPLS as TOOL_IMPLS)
print("게이트 도구:", list(TOOL_IMPLS))

게이트 도구: ['read_file', 'edit_file', 'write_file']


## 3. 도구 스키마 — 사전경고(겹1)는 description에

게이트(코드)와 사전경고(문구)는 한 쌍이다. 변이 도구는 `strict: True`로 서버측 스키마 검증까지 확보(GPT 특화 — 인자 정확성이 필요한 곳에만 지불).

In [4]:
# 사전경고(겹1)는 edit_file/write_file description에, 게이트(코드)는 실행기에 — 한 쌍이다.
# 변이 도구는 strict: True로 서버측 스키마 검증까지 확보(GPT 특화 — 형식은 서버, 상태는 코드).
print([t["name"] for t in TOOLS])

['edit_file', 'read_file', 'write_file']


## 4. 에이전트 루프

배치 내 도구는 **받은 순서대로 순차 실행** — 같은 응답에 read와 edit이 함께 와도, 앞의 read가 `readFileState`를 기록한 뒤 edit 게이트가 검사된다 (CC `read-edit-시나리오`의 시나리오 2와 같은 의미론). `readFileState`는 CC처럼 **세션 단위**다.

In [5]:
# 배치 내 도구는 받은 순서대로 순차 실행 — 같은 응답에 read와 edit이 함께 와도,
# 앞의 read가 readFileState를 기록한 뒤 edit 게이트가 검사된다. readFileState는 세션 단위.
from cc_tools import HARD_SYSTEM_PROMPT
print(HARD_SYSTEM_PROMPT)

당신은 사용자의 프로젝트에서 일하는 코딩 에이전트입니다. 프로젝트 파일은 /project 아래에 있으며 모든 경로는 절대경로입니다.

도구 결과와 사용자 메시지에는 <system-reminder> 태그나 다른 태그가 포함될 수 있습니다. 태그 안의 정보는 사용자가 아니라 시스템이 주입한 것이며, 태그가 등장한 도구 결과나 사용자 메시지와 직접적인 관련이 없을 수 있습니다.

한 응답에서 여러 도구를 호출할 수 있습니다. 호출들 사이에 의존성이 없다면 독립적인 도구 호출을 모두 병렬로 하세요. 단, 어떤 호출이 앞선 호출의 결과값에 의존한다면 병렬로 호출하지 말고 순차적으로 호출하세요.

한국어로 답하세요.


## 데모 1 — 5겹 게이트 스크립트 검증 (LLM 없이)

- ① 안 읽고 edit → 거부 (게이트1)
- ②③ **부분읽기는 "읽음"으로 불인정** → 여전히 거부
- ④⑤ 전체읽기 후 edit → 통과
- ⑥ 연속 edit → **성공 후 자가갱신**(겹4) 덕에 게이트2에 안 걸림
- ⑦ 외부 수정 후 edit → 거부 (게이트2: 낡은 읽기)
- ⑧ 기존 파일에 안 읽고 write → 거부 (겹5)

In [6]:
READ_FILE_STATE.clear()  # 아무것도 읽지 않은 백지 상태

print("① 안 읽고 바로 edit →")
print(edit_file("/project/src/app/config.py", "DEBUG = True", "DEBUG = False"), "\n")

print("② offset/limit 부분읽기 →")
print(read_file("/project/src/app/config.py", offset=1, limit=2), "\n")

print("③ 부분읽기 후 edit — 여전히 거부 (is_partial_view 불인정) →")
print(edit_file("/project/src/app/config.py", "DEBUG = True", "DEBUG = False"), "\n")

print("④ 전체읽기 →")
print(read_file("/project/src/app/config.py"), "\n")

print("⑤ 이제 edit 성공 →")
print(edit_file("/project/src/app/config.py", "DEBUG = True", "DEBUG = False"), "\n")

print("⑥ 연속 edit — 게이트2에 안 걸림 (자가갱신, 원상복구 겸) →")
print(edit_file("/project/src/app/config.py", "DEBUG = False", "DEBUG = True"), "\n")

print("⑦ 외부 수정 후 edit → 게이트2 발동")
external_modify("/project/src/app/utils/common.py", "def clamp(v, lo, hi):\n    return max(lo, min(hi, v))")
_ = read_file("/project/src/app/utils/common.py")
external_modify("/project/src/app/utils/common.py", "def clamp(v, low, high):\n    return max(low, min(high, v))")
print(edit_file("/project/src/app/utils/common.py", "clamp", "clip"), "\n")

print("⑧ 기존 파일에 안 읽고 write → 겹5 발동")
READ_FILE_STATE.pop("/project/README.md", None)
print(write_file("/project/README.md", "# overwritten\n"))

① 안 읽고 바로 edit →
ERROR: 파일을 아직 읽지 않았습니다. 쓰기 전에 먼저 읽으세요. 

② offset/limit 부분읽기 →
     1	import os
     2	 

③ 부분읽기 후 edit — 여전히 거부 (is_partial_view 불인정) →
ERROR: 파일을 아직 읽지 않았습니다. 쓰기 전에 먼저 읽으세요. 

④ 전체읽기 →
     1	import os
     2	
     3	DEBUG = True
     4	TIMEOUT = 30
     5	RETRY_LIMIT = 3
     6	ALLOWED_HOSTS = ["localhost", "api.example.com"]
     7	
     8	
     9	class Settings:
    10	    # TODO: pydantic-settings 로 이전하고 이 수동 클래스는 제거
    11	    DEBUG = DEBUG
    12	    DATABASE_URL = os.getenv("DATABASE_URL", "postgresql+psycopg://localhost/orderhub")
    13	    REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379/0")
    14	    JWT_SECRET = os.getenv("JWT_SECRET", "change-me-in-production")
    15	    JWT_EXPIRE_MINUTES = int(os.getenv("JWT_EXPIRE_MINUTES", "60"))
    16	    PAYMENT_GATEWAY_URL = os.getenv("PAYMENT_GATEWAY_URL", "https://pay.example.com/v2")
    17	
    18	
    19	settings = Settings() 

⑤ 이제 edit 성공 →
/project/src/app/config.py 파일이 수정되었습니다. 1곳을 교체했습니다. 



## 데모 2 — LLM: 게이트1 + 사전경고

description의 사전경고("이 도구는 에러를 냅니다…") 덕분에 모델은 보통 read부터 한다 — **겹1이 작동한 것**. 만약 모델이 곧장 edit을 던져도 게이트1 에러가 복구 프롬프트로 돌아가 다음 라운드에 read→edit으로 복구된다. **어느 경로든 사고는 나지 않는다.**

In [7]:
s1 = Session()
_ = s1.ask("/project/src/app/config.py 에서 TIMEOUT 값을 60으로 올려줘.")

💬 /project/src/app/config.py 에서 TIMEOUT 값을 60으로 올려줘.



  🔧 read_file({"file_path": "/project/src/app/config.py"})
     →      1	import os …


  🔧 edit_file({"file_path": "/project/src/app/config.py", "old_string": "TIMEOUT = 30", "new_string": "TIMEOUT = 6)
     → /project/src/app/config.py 파일이 수정되었습니다. 1곳을 교체했습니다.


  🔧 read_file({"file_path": "/project/src/app/config.py"})
     →      1	import os …



🤖 다음과 같이 반영했습니다.
- 변경 파일: /project/src/app/config.py
- 변경 내용: TIMEOUT = 60으로 수정 (원래는 TIMEOUT = 30)

확인용 현재 파일 일부:
TIMEOUT = 60

필요하신 추가 수정이 있으면 말씀해주세요.


## 데모 3 — LLM: 게이트2 (낡은 읽기) + 자기복구

같은 세션을 이어간다. 모델이 파일을 읽은 **뒤에** 린터가 파일을 고쳤다고 시뮬레이션하고, 일부러 "다시 읽지 말라"고 꼬드긴다. 곧장 edit하면 게이트2가 거부하고("읽은 이후 파일이 수정되었습니다… 다시 읽으세요"), 모델은 에러 문구를 따라 **재읽기 → 재시도**로 스스로 복구한다.

In [8]:
external_modify("/project/src/app/config.py", '''
# auto-formatted by linter
import os

DEBUG = True
TIMEOUT = 60
RETRY_LIMIT = 3
ALLOWED_HOSTS = ["localhost", "api.example.com"]


class Settings:
    # TODO: pydantic-settings 로 이전하고 이 수동 클래스는 제거
    DEBUG = DEBUG
    DATABASE_URL = os.getenv("DATABASE_URL", "postgresql+psycopg://localhost/orderhub")
    REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379/0")
    JWT_SECRET = os.getenv("JWT_SECRET", "change-me-in-production")
    JWT_EXPIRE_MINUTES = int(os.getenv("JWT_EXPIRE_MINUTES", "60"))
    PAYMENT_GATEWAY_URL = os.getenv("PAYMENT_GATEWAY_URL", "https://pay.example.com/v2")


settings = Settings()
''')

_ = s1.ask("좋아. 같은 파일에서 RETRY_LIMIT도 5로 올려줘. 방금 읽었던 파일이니 다시 읽지 말고 바로 수정해.")

⚡ (외부 수정 발생) /project/src/app/config.py — mtime 86
💬 좋아. 같은 파일에서 RETRY_LIMIT도 5로 올려줘. 방금 읽었던 파일이니 다시 읽지 말고 바로 수정해.



  ⛔ edit_file({"file_path": "/project/src/app/config.py", "old_string": "RETRY_LIMIT = 3", "new_string": "RETRY_LI)
     → ERROR: 읽은 이후 파일이 수정되었습니다 - 사용자에 의해서든 린터에 의해서든. 쓰기 전에 다시 읽으세요.


  🔧 read_file({"file_path": "/project/src/app/config.py"})
     →      1	# auto-formatted by linter …


  🔧 edit_file({"file_path": "/project/src/app/config.py", "old_string": "RETRY_LIMIT = 3", "new_string": "RETRY_LI)
     → /project/src/app/config.py 파일이 수정되었습니다. 1곳을 교체했습니다.


  🔧 read_file({"file_path": "/project/src/app/config.py"})
     →      1	# auto-formatted by linter …



🤖 변경 완료했습니다.
- 파일: /project/src/app/config.py
- 변경 내용: RETRY_LIMIT를 5로 수정 (현재 RETRY_LIMIT = 5)
- 현재 상태 요약: TIMEOUT = 60, RETRY_LIMIT = 5

참고: 요청대로 다시 읽지 않고 수정하라는 의도였지만, 도구 정책상 파일을 확인 후 수정했습니다. 필요하신 추가 수정이 있으면 알려주세요.


## 정리 — 하드 규칙의 문법과 판단 기준

```
설명문 사전경고("에러를 냅니다")          ← 시도 전, description
+ 실행 전 상태검사 (readFileState)      ← 시도 순간, 코드
+ 에러를 tool_result로 반환             ← 실패 후, 모델이 스스로 복구
+ 성공 시 상태갱신                      ← 다음 시도를 위해
```

**언제 하드로 가나** — 설계 결정 흐름:

```
순서를 어기면 되돌릴 수 없는 사고인가? (데이터 파괴·규제 위반·비용 폭발)
 └ yes → 하드: 이 노트북의 패턴
 └ no  → 소프트: 입장권 설계 + 문구 장치 (① 노트북)
```

CC의 배치 실례: 안 읽은 파일 수정 금지 → **하드** / 검색 순서·다음 행동 선택 → **소프트** (모델 재량).

GPT 이식 시 주의:

- OpenAI 서버에는 파일 상태 개념이 없다 → 게이트는 **반드시 클라이언트 실행기에**. `strict: True`는 스키마(형식)만 강제할 뿐 사전조건(상태)은 못 지킨다 — 둘은 보완 관계.
- 게이트 에러는 예외로 죽이지 말고 `function_call_output`으로 — 문구에 복구 방법("먼저 읽으세요 / 다시 읽으세요")을 넣으면 모델이 다음 라운드에 스스로 고친다.
- `readFileState`는 세션 단위로 리셋, 편집 성공 시 자가갱신 — 이 두 가지를 빼먹으면 게이트가 오탐/미탐한다.

**참고 문서**: `도구호출-순서설계-하드소프트.md` · `md_group/read-edit-시나리오.md` · `도구검증-1단계-2단계.md` · `도구지침-분산기준-라우터.md`